# Reasoning trace distillation: Colab run
Run the cells top to bottom. Every cell prints its output here, every command is saved to `logs/runs/`, failures are written to `logs/ERROR_LOG.md`, and every figure is drawn by `reporting/make_figures.py` from a data file (nothing typed in by hand).

One-time setup: in Colab, open the key icon (Secrets), add `GITHUB_TOKEN` (a GitHub personal access token with repo access) and switch on notebook access. Put your trace files in Google Drive at `MyDrive/reasoning-distillation/data/`.

## 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'dedeepyakm/reasoning-trace-distillation'
if not os.path.exists('/content/repo'):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/{REPO}.git','/content/repo'],check=True)
    subprocess.run(['git','-C','/content/repo','remote','set-url','origin',f'https://github.com/{REPO}.git'],check=True)
else:
    subprocess.run(['git','-C','/content/repo','pull',f'https://{TOKEN}@github.com/{REPO}.git','main'],check=True)
DRIVE_DATA = '/content/drive/MyDrive/reasoning-distillation/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
if not os.path.exists('/content/repo/data'):
    os.symlink(DRIVE_DATA, '/content/repo/data')
os.chdir('/content/repo')
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)
print(os.listdir('data'))

In [ ]:
!pip -q install transformers accelerate datasets matplotlib
!nvidia-smi

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
from common.run_logging import run_step, log_error, log_result
PY = 'python'
INSTRUCT_TRACES  = 'data/traces_teacher_k5.jsonl'
REASONING_TRACES = 'data/traces_reasoning_teacher_k5_v2_clean.jsonl'

## 2. Cut the traces and count (no GPU needed)

In [ ]:
run_step('analyze_reasoning', [PY,'analysis/analyze_traces_v3.py','--trace-file',REASONING_TRACES,'--out','data/cut_reasoning_final.jsonl','--with-difficulty'])

In [ ]:
run_step('analyze_instruct', [PY,'analysis/analyze_traces_v3.py','--trace-file',INSTRUCT_TRACES,'--out','data/cut_instruct_final.jsonl','--with-difficulty'])

## 3. Compare the two problem sets

In [ ]:
run_step('compare_problem_sets', [PY,'diagnostics/compare_problem_sets.py','--a','data/cut_instruct_final.jsonl','--b','data/cut_reasoning_final.jsonl'])

## 4. Score signal retention (uses the GPU; slow)
Set `RUN_SCORING = True` to run it.

In [ ]:
RUN_SCORING = False
if RUN_SCORING:
    run_step('score_reasoning', [PY,'scoring/score_cut_v2.py','--cut-file','data/cut_reasoning_final.jsonl','--trace-file',REASONING_TRACES,'--out','data/score_reasoning_final.json'])
    run_step('score_instruct',  [PY,'scoring/score_cut_v2.py','--cut-file','data/cut_instruct_final.jsonl','--trace-file',INSTRUCT_TRACES,'--out','data/score_instruct_final.json'])

## 5. Figures (drawn from the files above)

In [ ]:
for label in ['reasoning','instruct']:
    cmd = [PY,'reporting/make_figures.py','--cut-file',f'data/cut_{label}_final.jsonl','--label',label,'--outdir','data/figures']
    sc = f'data/score_{label}_final.json'
    if os.path.exists(sc):
        cmd += ['--score-file', sc]
    run_step('figures_'+label, cmd)

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/figures/*.png')):
    print(p)
    display(Image(p))

## 6. Logs

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 80)
pd.read_csv('logs/RESULT_LOG.csv').tail(40)

In [ ]:
print(open('logs/ERROR_LOG.md').read()[-3000:])

## 7. Save the logs to GitHub
The logs (not the data) go into the repo. Review `git status` first.

In [ ]:
subprocess.run(['git','add','logs'])
print(subprocess.run(['git','status','--short'],capture_output=True,text=True).stdout)

In [ ]:
COMMIT_MESSAGE = 'Update logs'
subprocess.run(['git','-c','user.name=Dedeepya Korukonda','-c','user.email=korukondadedeepya2002@gmail.com','commit','-m',COMMIT_MESSAGE])
subprocess.run(['git','push',f'https://{TOKEN}@github.com/{REPO}.git','main'])